<a href="https://colab.research.google.com/github/the1975/python_learning/blob/main/Data_Processing_scrapping_harga.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SCRAPPING**

**1. import library**

In [ ]:
import csv              # Untuk mengolah file Excel/CSV
import os               # Untuk akses sistem file/folder
import time             # Untuk mengatur jeda waktu (delay)
from dataclasses import dataclass  # Untuk bikin struktur data objek dengan mudah
from datetime import datetime      # Untuk mencatat tanggal dan waktu
from typing import Optional        # Untuk penanda tipe data yang boleh kosong (None)
from google.colab import userdata  # Untuk ambil API Key yang disimpan aman di Colab
import requests         # Untuk mengambil data dari internet/API

**2. konfigurasi**

In [ ]:
# konfigurasi atau setting
API_KEY = userdata.get('SerpAPI')  # Mengambil API Key SerpAPI yang disimpan di Colab
OUTPUT_FILE = "Data_iphone_Ecommerce.csv"  # Nama file CSV untuk menyimpan hasil scraping

# Keyword list yang mau di scrapping
KEYWORD_LIST = [
    "iphone 15",
    "iphone 15 pro",
    "iphone 15 pro max",
    "iphone 14",
    "iphone 14 pro",
    "iphone 14 pro max",
    "iphone 13",
    "iphone 13 pro",
    "iphone 13 pro max",
    "iphone 12",
    "iphone 12 pro",
    "iphone 12 pro max",
    "iphone 11",
    "iphone 11 pro",
    "iphone 11 pro max",
]

PAGES_PER_KEYWORD = 2          # Jumlah halaman pencarian yang diambil per kata kunci
RESULTS_PER_PAGE = 100         # Jumlah produk yang diambil per halaman
MIN_VALID_PRICE = 1_000_000    # Batas harga minimal Rp 1 Juta (untuk menyaring aksesoris)
REQUEST_DELAY_SECONDS = 1      # Jeda waktu (1 detik) tiap request agar tidak diblokir
REQUEST_TIMEOUT_SECONDS = 30   # Batas waktu maksimal menunggu respon web (30 detik)

# Judul kolom untuk tabel file CSV hasil akhir
CSV_HEADER = ["No", "Tanggal", "Nama Produk", "Harga_IDR", "Toko", "Link"]

**3. data class produk**

In [ ]:
# Model data
@dataclass
class Produk:
    """Representasi satu baris data produk yang akan disimpan ke CSV."""
    nomor: int       # Kolom nomor urut
    tanggal: str     # Kolom tanggal pengambilan data
    nama: str        # Kolom nama produk iPhone
    harga: float     # Kolom harga produk (angka)
    toko: str        # Kolom nama toko penjual
    link: str        # Kolom link/URL produk

    # Fungsi untuk mengubah data objek di atas menjadi bentuk List [ ] agar bisa ditulis ke CSV
    def to_row(self) -> list:
        return [self.nomor, self.tanggal, self.nama, self.harga, self.toko, self.link]

**4. csv helper function**

In [ ]:
def muat_produk_tersimpan(filepath: str) -> set[str]:
    """Baca CSV yang sudah ada, kembalikan set nama produk (lowercase) yang sudah tercatat."""
    nama_produk_tersimpan = set()  # Wadah kosong berbentuk 'set' untuk menyimpan nama produk unik

    # Jika file CSV belum pernah dibuat/tidak ada, langsung kembalikan set kosong
    if not os.path.exists(filepath):
        return nama_produk_tersimpan

    # Buka file CSV lama dengan mode 'r' (read/baca)
    with open(filepath, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None)  # Melewati baris pertama (judul kolom/header) agar tidak ikut terbaca

        # Looping setiap baris di file CSV lama
        for row in reader:
            if len(row) > 2:  # Pastikan kolom nama produk (kolom indeks ke-2) ada isinya
                nama_produk_tersimpan.add(row[2].lower())  # Simpan nama produk dalam huruf kecil semua

    return nama_produk_tersimpan


def tulis_header_jika_baru(writer: "csv.writer", jumlah_data_tersimpan: int) -> None:
    """Tulis header CSV hanya kalau file masih kosong/baru."""
    # Jika belum ada data lama (file baru dibuat), tulis baris judul kolom di paling atas
    if jumlah_data_tersimpan == 0:
        writer.writerow(CSV_HEADER)

**5. price parsing and validation**

In [ ]:
def ekstrak_harga(item: dict) -> Optional[float]:
    """
    Ambil harga numerik dari satu item hasil SerpAPI.
    SerpAPI biasanya sudah menyediakan 'extracted_price' (angka bersih),
    tapi kalau kosong, fallback ke parsing manual dari field 'price' (string "Rp1.234.000").
    """
    harga = item.get("extracted_price")  # Coba ambil harga yang sudah berbentuk angka bersih dari API
    if harga:
        return float(harga)  # Jika ada, langsung ubah ke format angka desimal (float)

    raw_price = item.get("price")  # Jika angka bersih tidak ada, ambil teks harga mentah (misal: "Rp 15.000.000")
    if not raw_price:
        return None  # Jika teks harga juga tidak ada, batalkan

    try:
        # Bersihkan teks dari simbol "Rp", titik, dan koma, lalu hilangkan spasi kosong
        cleaned = str(raw_price).replace("Rp", "").replace(".", "").replace(",", "").strip()
        return float(cleaned)  # Ubah teks yang sudah bersih menjadi angka desimal
    except ValueError:
        return None  # Jika gagal diubah jadi angka (error), kembalikan None


def harga_valid(harga: Optional[float]) -> bool:
    """Filter harga wajar untuk laptop, buang outlier/noise."""
    # Catatan: Di docstring tertulis 'laptop', tapi di konfigurasi awal untuk 'iPhone'
    # Memastikan harga tidak kosong dan nilainya di atas batas minimal (Rp 1 Juta)
    return harga is not None and harga > MIN_VALID_PRICE

**6. serpapi data retrieval**

In [ ]:
def build_search_url(keyword: str, start: int) -> str:
    """Susun URL request ke SerpAPI Google Shopping engine."""
    # Menyusun alamat URL lengkap beserta parameter pencarian (bahasa, negara, jumlah hasil, dll)
    return (
        f"https://serpapi.com/search.json?"
        f"engine=google&q={keyword}&tbm=shop&gl=id&hl=id"
        f"&num={RESULTS_PER_PAGE}&start={start}&api_key={API_KEY}"
    )


def fetch_shopping_results(keyword: str, page: int) -> Optional[list[dict]]:
    """
    Panggil SerpAPI untuk satu keyword + halaman tertentu.
    Return None kalau request gagal (network error atau status bukan 200).
    """
    start = page * RESULTS_PER_PAGE  # Menghitung index urutan produk awal untuk halaman ini
    url = build_search_url(keyword, start)  # Membuat URL tujuan berdasarkan kata kunci dan halamannya

    try:
        # Mengirim permintaan download data ke web target dengan batas waktu tunggu tertentu
        response = requests.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
    except requests.exceptions.RequestException as e:
        print(f"  -> Gagal koneksi: {e}")  # Tampilkan pesan jika internet putus atau timeout
        return None

    # Jika server merespon tapi kodenya bukan 200 (artinya error, misal API Key habis atau salah parameter)
    if response.status_code != 200:
        print(f"  -> Error API: {response.status_code}")
        return None

    # Mengubah format data mentah hasil download menjadi format JSON/Dictionary Python, lalu ambil list produknya
    return response.json().get("shopping_results", [])

**7. main srapping logic**

In [ ]:
def proses_keyword(
    keyword: str,
    produk_tersimpan: set[str],
    tanggal: str,
    writer: "csv.writer",
) -> int:
    """
    Scrape semua halaman untuk satu keyword, tulis produk baru ke CSV.
    Return jumlah produk baru yang ditambahkan untuk keyword ini.
    """
    print(f"Mencari: {keyword}...")
    jumlah_baru = 0  # Counter untuk menghitung berapa banyak produk baru yang didapat

    # Looping berdasarkan jumlah halaman yang ingin diambil (diatur di konfigurasi)
    for page in range(PAGES_PER_KEYWORD):
        hasil = fetch_shopping_results(keyword, page)  # Tarik data dari API

        if hasil is None:
            break  # Jika koneksi ke API gagal/error, langsung stop pencarian kata kunci ini
        if not hasil:
            break  # Jika halaman kosong (tidak ada produk lagi), stop pencarian kata kunci ini

        # Looping setiap produk yang ada di dalam halaman tersebut
        for item in hasil:
            nama = item.get("title", "Tanpa Judul")  # Ambil nama produk
            nama_lower = nama.lower()                # Ubah ke huruf kecil untuk pengecekan duplikat

            # Jika nama produk sudah pernah disimpan di file CSV sebelumnya, lewati produk ini
            if nama_lower in produk_tersimpan:
                continue

            harga = ekstrak_harga(item)  # Ambil angka harganya
            # Jika harga tidak valid (misal di bawah Rp 1 juta), lewati produk ini
            if not harga_valid(harga):
                continue

            # Jika lolos semua filter, bungkus data ke dalam objek/model 'Produk'
            produk = Produk(
                nomor=len(produk_tersimpan) + 1,  # Nomor urut otomatis (melanjutkan total data lama)
                tanggal=tanggal,
                nama=nama,
                harga=harga,
                toko=item.get("source", "Toko Tidak Tahu"),  # Ambil nama toko ecommerce
                link=item.get("link", ""),                    # Ambil URL/Link produk
            )

            writer.writerow(produk.to_row())  # Tulis langsung data produk baru ini ke file CSV
            produk_tersimpan.add(nama_lower)  # Masukkan nama produk ke daftar pelacak agar tidak duplikat nanti
            jumlah_baru += 1                  # Tambah hitungan produk baru

        # Tampilkan laporan progress setiap selesai memproses satu halaman
        print(f"  -> Halaman {page + 1} OK. Total database unik: {len(produk_tersimpan)}")
        time.sleep(REQUEST_DELAY_SECONDS)  # Beri jeda waktu sesuai konfigurasi agar aman dari blokir

    return jumlah_baru  # Kembalikan total jumlah data baru yang berhasil ditambahkan

**8. program execution**

In [ ]:
def main() -> None:
    # Mengambil tanggal hari ini dengan format Tahun-Bulan-Tanggal (YYYY-MM-DD)
    tanggal_hari_ini = datetime.now().strftime("%Y-%m-%d")

    # Memuat daftar nama produk yang sudah pernah disimpan sebelumnya di file CSV
    produk_tersimpan = muat_produk_tersimpan(OUTPUT_FILE)

    print("Memulai scraping: ")
    print("-" * 50)

    total_baru = 0  # Counter untuk menghitung total semua produk baru dari semua keyword

    # Membuka file CSV dengan mode 'a' (append) untuk menambah data baru di baris bawah tanpa menghapus data lama
    with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        # Tulis baris judul kolom (header) jika file CSV ini baru dibuat/masih kosong
        tulis_header_jika_baru(writer, len(produk_tersimpan))

        # Looping untuk memproses satu per satu kata kunci (keyword) yang ada di daftar konfigurasi
        for keyword in KEYWORD_LIST:
            # Jalankan fungsi pencarian dan akumulasikan jumlah produk baru yang didapat
            total_baru += proses_keyword(keyword, produk_tersimpan, tanggal_hari_ini, writer)

    # Menampilkan laporan akhir setelah semua kata kunci selesai diproses
    print("-" * 50)
    print(f"Selesai! Berhasil menambah {total_baru} data baru.")
    print(f"Total keseluruhan data di '{OUTPUT_FILE}': {len(produk_tersimpan)} baris.")


# Kode di bawah ini memastikan fungsi main() hanya berjalan ketika file script ini dieksekusi langsung
if __name__ == "__main__":
    main()

Memulai scraping: 
--------------------------------------------------
Mencari: iphone 15...
  -> Halaman 1 OK. Total database unik: 40
  -> Halaman 2 OK. Total database unik: 44
Mencari: iphone 15 pro...
  -> Halaman 1 OK. Total database unik: 76
  -> Halaman 2 OK. Total database unik: 89
Mencari: iphone 15 pro max...
  -> Halaman 1 OK. Total database unik: 118
  -> Halaman 2 OK. Total database unik: 127
Mencari: iphone 14...
  -> Halaman 1 OK. Total database unik: 165
Mencari: iphone 14 pro...
  -> Halaman 1 OK. Total database unik: 202
  -> Halaman 2 OK. Total database unik: 221
Mencari: iphone 14 pro max...
  -> Halaman 1 OK. Total database unik: 258
  -> Halaman 2 OK. Total database unik: 278
Mencari: iphone 13...
Mencari: iphone 13 pro...
  -> Halaman 1 OK. Total database unik: 317
Mencari: iphone 13 pro max...
  -> Halaman 1 OK. Total database unik: 352
  -> Halaman 2 OK. Total database unik: 354
Mencari: iphone 12...
  -> Halaman 1 OK. Total database unik: 393
  -> Halaman 2 OK.

# **PREPROCESSING**

**1. import library**

In [ ]:
# import library untuk persiapan EDA
import pandas as pd  # Library untuk manipulasi data (membuat tabel/DataFrame)
import numpy as np   # Library untuk operasi matematika dan manipulasi array/angka

# load dataset dan tampilkan 5 data teratas
df = pd.read_csv('Data_iphone_Ecommerce.csv')  # Membaca file hasil scraping menjadi tabel data (DataFrame)
df.head()                                      # Menampilkan 5 baris pertama untuk cek struktur datanya

,No,Tanggal,Nama Produk,Harga_IDR,Toko,Link
0,1,2026-06-30,Apple iPhone 15,12499000.0,Digimap,NaN
1,2,2026-06-30,"Apple iPhone 15 256GB, Pink",14749000.0,Shopee,NaN
2,3,2026-06-30,Apple Iphone 15 128GB I 256GB I 512GB Garansi ...,16499000.0,tokopedia.com,NaN
3,4,2026-06-30,Apple iPhone 15 Black,7315773.0,eBay,NaN
4,5,2026-06-30,"Apple iPhone 15 128GB, Black",12499000.0,Shopee,NaN


**2. cek nama kolom dan tipe data**

In [ ]:
# mengecek nama kolom dan tipe data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 603 entries, 0 to 602
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   No           603 non-null    int64  
 1   Tanggal      603 non-null    object 
 2   Nama Produk  603 non-null    object 
 3   Harga_IDR    603 non-null    float64
 4   Toko         603 non-null    object 
 5   Link         0 non-null      float64
dtypes: float64(2), int64(1), object(3)
memory usage: 28.4+ KB


**3. cek missing value**

In [ ]:
# mengecek missing value
df.isnull().sum().sort_values(ascending=False)

,0
Link,603
No,0
Tanggal,0
Nama Produk,0
Harga_IDR,0
Toko,0


**4. data cleaning**

In [ ]:
# Cleaning Data iPhone E-Commerce (dengan deteksi outlier)
# import library yang dibutuhkan

import pandas as pd  # Untuk manipulasi dan analisis data tabel
import re           # Untuk pencarian teks menggunakan pola (Regular Expression)

# ============================================================
# 1. LOAD & CLEANING DASAR
# ============================================================
# Membaca file mentah hasil scraping
df = pd.read_csv("Data_iphone_Ecommerce.csv")

# Hapus baris pertama karena biasanya berisi judul kolom yang double/duplikat
df = df.drop(0).reset_index(drop=True)

# Mengubah kolom harga dari teks (object) menjadi angka (numeric) agar bisa dihitung
df["Harga_IDR"] = pd.to_numeric(df["Harga_IDR"])

# Membuat kolom baru: konversi Rupiah ke satuan Juta (misal: 15.000.000 jadi 15.0)
df["Harga_Juta"] = df["Harga_IDR"] / 1000000

# Menghapus kolom nomor urut lama dan link karena tidak dipakai dalam analisis
df = df.drop(columns=["No", "Link"])


# ============================================================
# 2. EKSTRAKSI SERI & VARIAN
# ============================================================

def extract_seri(nama: str) -> str:
    """Ambil seri iPhone dari nama produk, misal 'iPhone 14'."""
    # Cari pola kata 'iphone' yang diikuti oleh 2 digit angka (misal: iphone 11, iphone 15)
    m = re.search(r"iphone\s*(\d{2})", nama, re.IGNORECASE)
    return f"iPhone {m.group(1)}" if m else "Lainnya"


def extract_varian(nama: str) -> str:
    """Tentukan varian: Pro Max, Pro, Plus, atau Biasa."""
    n = nama.lower()
    if "pro max" in n or "promax" in n or "pro  max" in n:
        return "Pro Max"
    elif "pro" in n:
        return "Pro"
    elif "plus" in n:
        return "Plus"
    return "Biasa"  # Jika tidak mengandung kata di atas, dianggap versi standar/biasa


# Menjalankan fungsi ekstrak di atas untuk membuat kolom 'Seri' dan 'Varian' baru
df["Seri"] = df["Nama Produk"].apply(extract_seri)
df["Varian"] = df["Nama Produk"].apply(extract_varian)


# ============================================================
# 3. DETEKSI OUTLIER HARGA (IQR per kategori Seri + Varian)
# ============================================================

def flag_outlier(group: pd.DataFrame) -> pd.Series:
    # Cari titik batas bawah (25%) dan batas atas (75%) dari data harga
    q1 = group["Harga_Juta"].quantile(0.25)
    q3 = group["Harga_Juta"].quantile(0.75)
    iqr = q3 - q1  # Jarak antar kuartil (Interquartile Range)

    # Hitung batas toleransi harga wajar (menggunakan rumus ekstrem: 3x IQR)
    batas_bawah = q1 - 3 * iqr
    batas_atas = q3 + 3 * iqr

    # Berikan tanda True jika harga terlalu murah atau terlalu mahal (outlier)
    return (group["Harga_Juta"] < batas_bawah) | (group["Harga_Juta"] > batas_atas)


# Kelompokkan data berdasarkan Seri & Varian, lalu cari data yang harganya tidak wajar
df["is_outlier"] = df.groupby(["Seri", "Varian"], group_keys=False).apply(flag_outlier)

# Menghitung dan menampilkan total baris data yang dianggap outlier
jumlah_outlier = df["is_outlier"].sum()
print(f"Outlier terdeteksi: {jumlah_outlier} dari {len(df)} baris ({jumlah_outlier/len(df)*100:.1f}%)")
print()
print("Daftar listing yang ke-flag sebagai outlier (cek manual sebelum dipercaya 100%):")

# Lebarkan tampilan kolom teks di terminal/notebook agar nama produk tidak terpotong
pd.set_option("display.max_colwidth", 80)

# Tampilkan data outlier saja, diurutkan dari yang harganya paling mahal
print(
    df[df["is_outlier"]][["Nama Produk", "Harga_Juta", "Toko", "Seri", "Varian"]]
    .sort_values("Harga_Juta", ascending=False)
    .to_string()
)


# ============================================================
# 4. SIMPAN CSV CLEANED
# ============================================================
# Menyimpan hasil tabel yang sudah bersih ke file CSV baru tanpa menyertakan kolom indeks bawaan pandas
df.to_csv("Data_iphone_e_commerce_cleaned.csv", index=False)

print()
print(f"Disimpan: Data_iphone_e_commerce_cleaned.csv ({len(df)} baris, {len(df.columns)} kolom)")
print(f"   Kolom: {', '.join(df.columns)}")

# Menampilkan 5 baris pertama data hasil pembersihan
df.head()

Outlier terdeteksi: 23 dari 602 baris (3.8%)

Daftar listing yang ke-flag sebagai outlier (cek manual sebelum dipercaya 100%):
                                                                                                                          Nama Produk  Harga_Juta           Toko       Seri   Varian
242  Apple iPhone 14 Pro Max 5G 6.7" eSIM 256GB ROM 6GB RAM Genuine Retina OLED Face ID NFC A15 14ProMax US Version 98%New Smartphone   95.192000  tokopedia.com  iPhone 14  Pro Max
52                    Apple iPhone 15 Pro Max Titanium Series 128 / 256 / 512 GB / 1TB Original Product - White Titanium, Pro Max 1TB   69.131000  tokopedia.com  iPhone 15  Pro Max
515                                                              Apple iPhone 11 Pro Max - 512GB - Gold (Unlocked) A2161 (CDMA + GSM)   64.456569     ubuy.co.id  iPhone 11  Pro Max
338                                                                                 Apple iPhone 13 Pro Max - 1TB - Silver (Unlocked)   62.345989    

/tmp/ipykernel_1477/147781830.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["is_outlier"] = df.groupby(["Seri", "Varian"], group_keys=False).apply(flag_outlier)


,Tanggal,Nama Produk,Harga_IDR,Toko,Harga_Juta,Seri,Varian,is_outlier
0,2026-06-30,"Apple iPhone 15 256GB, Pink",14749000.0,Shopee,14.749000,iPhone 15,Biasa,False
1,2026-06-30,"Apple Iphone 15 128GB I 256GB I 512GB Garansi Resmi iBox TAM - Promo Random,...",16499000.0,tokopedia.com,16.499000,iPhone 15,Pro,False
2,2026-06-30,Apple iPhone 15 Black,7315773.0,eBay,7.315773,iPhone 15,Biasa,False
3,2026-06-30,"Apple iPhone 15 128GB, Black",12499000.0,Shopee,12.499000,iPhone 15,Biasa,False
4,2026-06-30,HP Apple iPhone 15 PRO/15 PRO MAX 64/128/256GB/512GB Fullset Second 100% Ori...,3062500.0,tokopedia.com,3.062500,iPhone 15,Pro Max,False


# **VISUALISASI**

In [ ]:
# mengimport library yang dibutuhkan untuk visualisasi data
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# load data yang udah di cleaning
df_visualisasi = pd.read_csv('Data_iphone_e_commerce_cleaned.csv')

# tampilkan 5 data teratas
df_visualisasi.head()

,Tanggal,Nama Produk,Harga_IDR,Toko,Harga_Juta,Seri,Varian,is_outlier
0,2026-06-30,"Apple iPhone 15 256GB, Pink",14749000.0,Shopee,14.749000,iPhone 15,Biasa,False
1,2026-06-30,"Apple Iphone 15 128GB I 256GB I 512GB Garansi Resmi iBox TAM - Promo Random,...",16499000.0,tokopedia.com,16.499000,iPhone 15,Pro,False
2,2026-06-30,Apple iPhone 15 Black,7315773.0,eBay,7.315773,iPhone 15,Biasa,False
3,2026-06-30,"Apple iPhone 15 128GB, Black",12499000.0,Shopee,12.499000,iPhone 15,Biasa,False
4,2026-06-30,HP Apple iPhone 15 PRO/15 PRO MAX 64/128/256GB/512GB Fullset Second 100% Ori...,3062500.0,tokopedia.com,3.062500,iPhone 15,Pro Max,False


In [ ]:
# @title Dashboard Visualisasi Harga iPhone E-Commerce { run: "auto" }


# ============================================================
# CELL 0 — Load & bersihkan data (jalankan sekali di awal)
# ============================================================
import pandas as pd
import re

df_raw = pd.read_csv("Data_iphone_e_commerce_cleaned.csv")

# NOTE: kolom Seri, Varian, dan is_outlier seharusnya SUDAH ADA di CSV ini
# kalau Tuan pakai script cleaning_iphone_dengan_outlier.py. Kalau belum
# ada, di bawah ini ada fallback yang generate ulang seadanya.


def extract_seri(nama: str) -> str:
    """Ambil seri iPhone dari nama produk, misal 'iPhone 14'."""
    m = re.search(r"iphone\s*(\d{2})", nama, re.IGNORECASE)
    return f"iPhone {m.group(1)}" if m else "Lainnya"


def extract_varian(nama: str) -> str:
    """Tentukan varian: Pro Max, Pro, Plus, atau Biasa."""
    n = nama.lower()
    if "pro max" in n or "promax" in n or "pro  max" in n:
        return "Pro Max"
    elif "pro" in n:
        return "Pro"
    elif "plus" in n:
        return "Plus"
    return "Biasa"


def normalisasi_toko(toko: str) -> str:
    """Gabungkan variasi penulisan nama toko (tokopedia.com -> Tokopedia, dst)."""
    t = str(toko).lower()
    if "tokopedia" in t:
        return "Tokopedia"
    elif "shopee" in t:
        return "Shopee"
    elif "ebay" in t:
        return "eBay"
    return "Lainnya"


df_visualisasi = df_raw.copy()

if "Seri" not in df_visualisasi.columns:
    df_visualisasi["Seri"] = df_visualisasi["Nama Produk"].apply(extract_seri)
if "Varian" not in df_visualisasi.columns:
    df_visualisasi["Varian"] = df_visualisasi["Nama Produk"].apply(extract_varian)
if "is_outlier" not in df_visualisasi.columns:
    df_visualisasi["is_outlier"] = False
    print("Kolom 'is_outlier' tidak ada di CSV — jalankan dulu script")
    print("   cleaning_iphone_dengan_outlier.py supaya outlier ekstrem terdeteksi.")

df_visualisasi["Toko_Bersih"] = df_visualisasi["Toko"].apply(normalisasi_toko)

# Urutan seri biar konsisten tampil dari lama ke baru di semua chart
urutan_seri = ["iPhone 11", "iPhone 12", "iPhone 13", "iPhone 14", "iPhone 15", "iPhone 17", "Lainnya"]
urutan_seri = [s for s in urutan_seri if s in df_visualisasi["Seri"].unique()] + \
              [s for s in df_visualisasi["Seri"].unique() if s not in urutan_seri]
df_visualisasi["Seri"] = pd.Categorical(df_visualisasi["Seri"], categories=urutan_seri, ordered=True)

print(f"Total data dimuat : {len(df_visualisasi)} listing")
print(f"Seri ditemukan     : {', '.join(urutan_seri)}")
print(f"Toko ditemukan     : {', '.join(df_visualisasi['Toko_Bersih'].unique())}")


# ============================================================
# CELL 1 — Filter Dashboard Visualisasi (jalankan & ganti dropdown sesuka hati)
# ============================================================

import plotly.express as px
from IPython.display import display, HTML

# Dropdown pilihan seri iPhone
Nama_Seri = "iPhone 13"  # @param ["SEMUA", "iPhone 11", "iPhone 12", "iPhone 13", "iPhone 14", "iPhone 15"]

# Dropdown pilihan varian
Varian_Pilihan = "SEMUA"  # @param ["SEMUA", "Biasa", "Plus", "Pro", "Pro Max"]

# Dropdown pilihan toko
Toko_Pilihan = "Tokopedia"  # @param ["SEMUA", "Shopee", "Tokopedia", "eBay", "Lainnya"]

# Dropdown jumlah listing termurah/termahal yang mau ditampilkan di chart
Jumlah_Tampil = "15"  # @param ["5", "10", "15", "20", "30"]

# Dropdown: sertakan listing yang ke-flag sebagai outlier harga ekstrem?
Sertakan_Outlier = "Tidak"  # @param ["Tidak", "Ya"]

jumlah_n = int(Jumlah_Tampil)

# ---------- Terapkan filter ----------
df_f = df_visualisasi.copy()

if Sertakan_Outlier == "Tidak":
    n_sebelum = len(df_f)
    df_f = df_f[~df_f["is_outlier"]]
    n_outlier_dibuang = n_sebelum - len(df_f)
else:
    n_outlier_dibuang = 0

if Nama_Seri != "SEMUA":
    df_f = df_f[df_f["Seri"] == Nama_Seri]

if Varian_Pilihan != "SEMUA":
    df_f = df_f[df_f["Varian"] == Varian_Pilihan]

if Toko_Pilihan != "SEMUA":
    df_f = df_f[df_f["Toko_Bersih"] == Toko_Pilihan]

print("Sumber       : Tokopedia, Shopee, eBay, dkk (e-commerce)")
print(f"Filter seri  : {Nama_Seri}")
print(f"Filter varian: {Varian_Pilihan}")
print(f"Filter toko  : {Toko_Pilihan}")
print(f"Outlier      : {'disembunyikan (' + str(n_outlier_dibuang) + ' listing dibuang)' if Sertakan_Outlier == 'Tidak' else 'ditampilkan semua'}")
print(f"Total listing: {len(df_f)} produk\n")

if len(df_f) == 0:
    print("⚠️ Tidak ada data yang cocok dengan filter ini. Coba ganti filter di atas.")
else:

    # ========== KPI CARDS ==========
    total_listing = len(df_f)
    harga_termurah = df_f["Harga_Juta"].min()
    harga_termahal = df_f["Harga_Juta"].max()
    harga_rata2 = df_f["Harga_Juta"].mean()
    toko_terbanyak = df_f["Toko_Bersih"].value_counts().idxmax()

    display(HTML(f"""
    <div style="
        display: flex;
        justify-content: space-around;
        background: linear-gradient(135deg, #1a1a2e, #16213e);
        padding: 20px;
        border-radius: 14px;
        border: 1px solid #0f3460;
        margin-bottom: 24px;
        gap: 12px;
    ">
        <div style="text-align:center; flex:1;">
            <p style="margin:0; color:#94a3b8; font-size:13px; font-weight:bold; letter-spacing:1px;">
                TOTAL LISTING
            </p>
            <p style="margin:8px 0 0 0; color:#38bdf8; font-size:28px; font-weight:bold;">
                {total_listing}
            </p>
            <p style="margin:2px 0 0 0; color:#64748b; font-size:11px;">produk ditemukan</p>
        </div>

        <div style="text-align:center; flex:1; border-left:1px solid #0f3460; border-right:1px solid #0f3460; padding:0 20px;">
            <p style="margin:0; color:#94a3b8; font-size:13px; font-weight:bold; letter-spacing:1px;">
                HARGA TERMURAH
            </p>
            <p style="margin:8px 0 0 0; color:#4ade80; font-size:22px; font-weight:bold;">
                Rp {harga_termurah:,.2f} jt
            </p>
            <p style="margin:2px 0 0 0; color:#64748b; font-size:11px;">listing termurah</p>
        </div>

        <div style="text-align:center; flex:1; border-right:1px solid #0f3460; padding:0 20px;">
            <p style="margin:0; color:#94a3b8; font-size:13px; font-weight:bold; letter-spacing:1px;">
                HARGA TERMAHAL
            </p>
            <p style="margin:8px 0 0 0; color:#f59e0b; font-size:22px; font-weight:bold;">
                Rp {harga_termahal:,.2f} jt
            </p>
            <p style="margin:2px 0 0 0; color:#64748b; font-size:11px;">listing termahal</p>
        </div>

        <div style="text-align:center; flex:1;">
            <p style="margin:0; color:#94a3b8; font-size:13px; font-weight:bold; letter-spacing:1px;">
                RATA-RATA HARGA
            </p>
            <p style="margin:8px 0 0 0; color:#c084fc; font-size:28px; font-weight:bold;">
                Rp {harga_rata2:,.2f} jt
            </p>
            <p style="margin:2px 0 0 0; color:#64748b; font-size:11px;">toko terbanyak: {toko_terbanyak}</p>
        </div>
    </div>
    """))

    # ========== CHART 1: Bar — Rata-rata Harga per Seri & Varian ==========
    # Kenapa bar chart: membandingkan rata-rata harga antar kategori (seri x varian)
    # adalah perbandingan kategorikal klasik -> bar chart paling jelas dibaca.
    df_seri_varian = (
        df_f.groupby(["Seri", "Varian"], observed=True)["Harga_Juta"]
        .agg(rata_rata="mean", jumlah="count")
        .reset_index()
    )
    df_seri_varian = df_seri_varian[df_seri_varian["jumlah"] > 0]

    fig_bar_seri = px.bar(
        df_seri_varian,
        x="Seri",
        y="rata_rata",
        color="Varian",
        barmode="group",
        title="Rata-Rata Harga iPhone per Seri & Varian",
        labels={"rata_rata": "Rata-Rata Harga (Juta Rupiah)", "Seri": "Seri iPhone"},
        text=df_seri_varian["rata_rata"].round(1),
        category_orders={"Varian": ["Biasa", "Plus", "Pro", "Pro Max"]},
        color_discrete_map={
            "Biasa": "#38bdf8",
            "Plus": "#4ade80",
            "Pro": "#f59e0b",
            "Pro Max": "#c084fc",
        },
    )
    fig_bar_seri.update_traces(texttemplate="%{text} jt", textposition="outside")
    fig_bar_seri.update_layout(
        height=550,
        plot_bgcolor="#0f0f0f",
        paper_bgcolor="#1a1a1a",
        font_color="white",
        title_font_size=16,
        legend_title_text="Varian",
    )
    fig_bar_seri.show()



    # ========== CHART 3: Bar Horizontal — Rata-rata Harga per Toko ==========
    # Kenapa horizontal bar: membandingkan harga antar toko (kategori sedikit,
    # nama toko bisa panjang) -> horizontal bar lebih rapi dibaca dari bar vertikal.
    df_toko = (
        df_f.groupby("Toko_Bersih")["Harga_Juta"]
        .agg(rata_rata="mean", jumlah="count")
        .reset_index()
        .sort_values("rata_rata")
    )

    fig_toko = px.bar(
        df_toko,
        x="rata_rata",
        y="Toko_Bersih",
        orientation="h",
        title="Rata-Rata Harga iPhone per Toko",
        labels={"rata_rata": "Rata-Rata Harga (Juta Rupiah)", "Toko_Bersih": "Toko"},
        color="rata_rata",
        color_continuous_scale="Blues",
        text=df_toko["rata_rata"].round(1),
        hover_data={"jumlah": True},
    )
    fig_toko.update_traces(texttemplate="%{text} jt", textposition="outside")
    fig_toko.update_layout(
        height=400,
        coloraxis_showscale=False,
        plot_bgcolor="#0f0f0f",
        paper_bgcolor="#1a1a1a",
        font_color="white",
        title_font_size=16,
    )
    fig_toko.show()

    # ========== CHART 4: Strip Plot — Sebaran Harga vs Toko, per Listing ==========
    # Kenapa strip/scatter (bukan scatter dua-angka biasa): kita mau lihat
    # setiap listing individual tersebar di harga berapa pada tiap toko,
    # sekaligus highlight termurah & termahal -> strip plot pas untuk
    # "satu kategori vs satu nilai numerik" dengan banyak titik data.
    df_strip = df_f.copy()
    df_strip["Label_Singkat"] = df_strip["Nama Produk"].str.slice(0, 40) + "..."

    fig_strip = px.strip(
        df_strip,
        x="Toko_Bersih",
        y="Harga_Juta",
        color="Seri",
        title="Sebaran Harga per Listing, Dikelompokkan per Toko",
        labels={"Harga_Juta": "Harga (Juta Rupiah)", "Toko_Bersih": "Toko"},
        hover_name="Label_Singkat",
        hover_data={"Varian": True, "Harga_Juta": ":.2f", "Toko_Bersih": False},
        category_orders={"Seri": urutan_seri},
    )
    fig_strip.update_traces(marker=dict(size=7, opacity=0.7))
    fig_strip.update_layout(
        height=600,
        plot_bgcolor="#0f0f0f",
        paper_bgcolor="#1a1a1a",
        font_color="white",
        title_font_size=16,
        legend_title_text="Seri",
    )
    fig_strip.show()

    # ========== CHART 5: Top N Termurah & Termahal (Bar Horizontal) ==========
    # Kenapa bar chart lagi: ranking top-N listing termurah/termahal adalah
    # perbandingan nilai per-item -> horizontal bar paling mudah dibaca
    # urutannya dari atas ke bawah.
    df_termurah = df_f.nsmallest(jumlah_n, "Harga_Juta").sort_values("Harga_Juta", ascending=False).copy()
    df_termurah["Label_Singkat"] = df_termurah["Nama Produk"].str.slice(0, 45)

    fig_termurah = px.bar(
        df_termurah,
        x="Harga_Juta",
        y="Label_Singkat",
        orientation="h",
        color="Toko_Bersih",
        title=f"Top {jumlah_n} Listing Termurah",
        labels={"Harga_Juta": "Harga (Juta Rupiah)", "Label_Singkat": "Produk"},
        text=df_termurah["Harga_Juta"].round(2),
    )
    fig_termurah.update_traces(texttemplate="%{text} jt", textposition="outside")
    fig_termurah.update_layout(
        height=max(400, jumlah_n * 28),
        plot_bgcolor="#0f0f0f",
        paper_bgcolor="#1a1a1a",
        font_color="white",
        title_font_size=16,
        legend_title_text="Toko",
        yaxis=dict(tickfont=dict(size=10)),
    )
    fig_termurah.show()

    df_termahal = df_f.nlargest(jumlah_n, "Harga_Juta").sort_values("Harga_Juta").copy()
    df_termahal["Label_Singkat"] = df_termahal["Nama Produk"].str.slice(0, 45)

    fig_termahal = px.bar(
        df_termahal,
        x="Harga_Juta",
        y="Label_Singkat",
        orientation="h",
        color="Toko_Bersih",
        title=f"Top {jumlah_n} Listing Termahal",
        labels={"Harga_Juta": "Harga (Juta Rupiah)", "Label_Singkat": "Produk"},
        text=df_termahal["Harga_Juta"].round(2),
    )
    fig_termahal.update_traces(texttemplate="%{text} jt", textposition="outside")
    fig_termahal.update_layout(
        height=max(400, jumlah_n * 28),
        plot_bgcolor="#0f0f0f",
        paper_bgcolor="#1a1a1a",
        font_color="white",
        title_font_size=16,
        legend_title_text="Toko",
        yaxis=dict(tickfont=dict(size=10)),
    )
    fig_termahal.show()


# ============================================================
# Auto-scroll ke atas setelah render (khusus Google Colab)
# ============================================================
try:
    from google.colab import output

    output.eval_js("""
    setTimeout(() => {
      const start = window.scrollY;
      const duration = 1500;
      const startTime = performance.now();
      function easeOutCubic(t){ return 1 - Math.pow(1 - t, 3); }
      function animate(currentTime){
        const elapsed = currentTime - startTime;
        const progress = Math.min(elapsed / duration, 1);
        window.scrollTo(0, start * (1 - easeOutCubic(progress)));
        if(progress < 1){ requestAnimationFrame(animate); }
      }
      requestAnimationFrame(animate);
    }, 1500);
    """)
except ImportError:
    pass  # bukan di Colab, skip auto-scroll

Total data dimuat : 602 listing
Seri ditemukan     : iPhone 11, iPhone 12, iPhone 13, iPhone 14, iPhone 15, iPhone 17, Lainnya
Toko ditemukan     : Shopee, Tokopedia, eBay, Lainnya
Sumber       : Tokopedia, Shopee, eBay, dkk (e-commerce)
Filter seri  : iPhone 13
Filter varian: SEMUA
Filter toko  : Tokopedia
Outlier      : disembunyikan (23 listing dibuang)
Total listing: 34 produk

